<h1>NAO RMSE - SPREAD - SATURATION</h1>

![UFS-logo](../../../UFS-Logo-RGB-2csolidshorizontal-72dpi-min.png)

In [1]:
basedir = f'../../../..'

In [2]:
import os
import sys
import xarray as xr

# Point to root directory of repository
root_dir = os.path.join(os.getcwd(), basedir)
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)
    
from src.datareader import datareader as dr
from src.util import util, stats

import warnings
warnings.filterwarnings('ignore')

<h5>Get data readers</h5>

In [3]:
# seas5 data are here:
seas5_dir = '/groups/ORC-CLIMATE/aoes_repo/models/seas5/monthly/mean/mslp/'

In [4]:
# Collect all sst seas5 files in known directory.
seas5_file_list = os.listdir(seas5_dir)

# Prepend file paths.
seas5_file_list = [os.path.join(seas5_dir, this_file) for this_file in seas5_file_list]

In [5]:
# netcdf4 package is needed here.
seas5_ds = xr.open_mfdataset(seas5_file_list, engine='netcdf4')

In [6]:
# Rename dimensions/coordinates to match our init+lead paradigm.
seas5_ds = seas5_ds.rename_dims({'forecast_reference_time': 'init',
                                 'forecastMonth': 'lead',
                                  'number': 'member'})

seas5_ds = seas5_ds.rename({'forecast_reference_time': 'init',
                            'forecastMonth': 'lead',
                            'number': 'member'})

# By default the lead unit is defined as '1', but we know it's really 'months'.
seas5_ds['lead'].attrs['units'] = 'months'

# Wrap it up into a DataReader object.
seas5_data_reader = dr.getDataReader(datasource='SUPPLIED', dataset=seas5_ds)

In [7]:
ufs_experiments = ['baseline', 'beta.0.1', 'cpc_ics']

ufs_data_readers = [dr.getDataReader(datasource='UFS',
                                     # filename=f'experiments/phase_1/{m}/atm_monthly.zarr',
                                     experiment = this_experiment,
                                     model='atm')
                    for this_experiment in ufs_experiments]

No filename provided; deferring to default
Reading data from s3://noaa-oar-sfsdev-pds/experiments/phase_1/baseline/atm_monthly.zarr
No filename provided; deferring to default
Reading data from s3://noaa-oar-sfsdev-pds/experiments/phase_1/beta.0.1/atm_monthly.zarr
No filename provided; deferring to default
Reading data from s3://noaa-oar-sfsdev-pds/experiments/phase_1/cpc_ics/atm_monthly.zarr


In [8]:
era5_data_reader = dr.getDataReader(datasource='ERA5')

No filename provided; deferring to default
Reading data from gs://gcp-public-data-arco-era5/ar/1959-2022-6h-512x256_equiangular_conservative.zarr


In [9]:
ufs_vars = ['slp', 'prmsl']
era5_var = 'mean_sea_level_pressure'
seas5_var = 'msl'

In [10]:
# Enter a list of members, like [1, 2, 6, 8, ens_avg]
# Note that 'ens_avg' is a special keyword in the ensuing code.
# If you include 'ens_avg' in the list of members,
# then the Ensemble Average will be listed under member = -1
members = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 'ens_avg']

# We know that seas5 hindcast has members 0-24.
seas5_members = list(range(25)) + ['ens_avg']

<h5>Define time period</h5>

In [11]:
time_range = ("1994-01-01", "2021-12-31T23")
initmonths = (5,)

In [12]:
# For NAO, there are 2 reference locations:
region_1 = {'latmin': 65.0, 'lonmin': 331.2}
region_2 = {'latmin': 37.7, 'lonmin': 334.3}

In [13]:
# Get 2 ERA5 datasets
era5_ds_1 = era5_data_reader.retrieve(var=era5_var,
                lat=region_1['latmin'],
                lon=region_1['lonmin'],
                time=time_range).squeeze(['lat', 'lon']).load()  # flatten

era5_ds_2 = era5_data_reader.retrieve(var=era5_var,
                lat=region_2['latmin'],
                lon=region_2['lonmin'],
                time=time_range).squeeze(['lat', 'lon']).load()  # flatten



In [14]:
seas5_ds_1 = util.retrieve_ufs_dataset(seas5_data_reader, seas5_var, time_range,
                                       seas5_members, region_1, initmonths=initmonths).squeeze(['lat', 'lon']).load()

seas5_ds_2 = util.retrieve_ufs_dataset(seas5_data_reader, seas5_var, time_range,
                                       seas5_members, region_2, initmonths=initmonths).squeeze(['lat', 'lon']).load()

Taking Ensemble Average
Taking Ensemble Average


In [15]:
%%capture captured_output

ufs_dss_1 = []
ufs_dss_2 = []
ufs_var_list = []
ufs_stats_1 = []
ufs_stats_2 = []

for this_dr in ufs_data_readers:

    for this_var in ufs_vars:                                                                                   
        if this_var in list(this_dr.dataset().keys()):                                                               
            ufs_var = this_var

    ufs_var_list.append(ufs_var) 
    

    this_ds_1 = util.retrieve_ufs_dataset(this_dr, ufs_var, time_range,
                                          members, region_1, initmonths=initmonths).squeeze(['lat', 'lon'])
    
    this_ds_2 = util.retrieve_ufs_dataset(this_dr, ufs_var, time_range,
                                          members, region_2, initmonths=initmonths).squeeze(['lat', 'lon'])

    ufs_dss_1.append(this_ds_1.load())
    ufs_dss_2.append(this_ds_2.load())

    # Calculate climatology statistics
    ufs_stats_1.append(stats.calc_climatology_anomaly(ufs_dss_1[-1], area_mean=False))
    ufs_stats_2.append(stats.calc_climatology_anomaly(ufs_dss_2[-1], area_mean=False))

In [16]:
seas5_stats_1 = stats.calc_climatology_anomaly(seas5_ds_1, area_mean=False)
seas5_stats_2 = stats.calc_climatology_anomaly(seas5_ds_2, area_mean=False)

In [17]:
era5_stats_1 = stats.calc_climatology_anomaly(era5_ds_1, area_mean=False)
era5_stats_2 = stats.calc_climatology_anomaly(era5_ds_2, area_mean=False)

In [18]:
# Normalize UFS datasets
ufs_das_1 = []
ufs_das_2 = []

for i in range(len(ufs_dss_1)):  
    ufs_das_1.append(stats.normalize(da=ufs_dss_1[i][ufs_var_list[i]], stats=ufs_stats_1[i]))
    ufs_das_2.append(stats.normalize(da=ufs_dss_2[i][ufs_var_list[i]], stats=ufs_stats_2[i]))

In [19]:
# Normalize SEAS5 datasets
seas5_da_1 = stats.normalize(da=seas5_stats_1['monthly_mean'], stats=seas5_stats_1)
seas5_da_2 = stats.normalize(da=seas5_stats_2['monthly_mean'], stats=seas5_stats_2)

In [20]:
# Normalize VERIF datasets
era5_da_1 = stats.normalize(da=era5_stats_1['monthly_mean'], stats=era5_stats_1)
era5_da_2 = stats.normalize(da=era5_stats_2['monthly_mean'], stats=era5_stats_2)



<h2>Calculate NAO</h2>

In [21]:
ufs_nao = []

for i in range(len(ufs_das_1)):
    ufs_nao.append((ufs_das_2[i] - ufs_das_1[i]).to_dataset())

In [22]:
seas5_nao = (seas5_da_2 - seas5_da_1).to_dataset()

In [23]:
era5_nao = (era5_da_2 - era5_da_1).to_dataset()

<h2>NAO Climatology</h2>

In [24]:
ufs_stats = []

for i in range(len(ufs_nao)):
    ufs_stats.append(stats.calc_climatology_anomaly(ufs_nao[i], area_mean=False, use_member_climatology=True))

In [25]:
seas5_stats = stats.calc_climatology_anomaly(seas5_nao, area_mean=False, use_member_climatology=True)

In [26]:
era5_stats = stats.calc_climatology_anomaly(era5_nao, area_mean=False)

<h2>Accumulating RMSE and SPREAD statistics for each UFS model. This may take some time!</h2>

In [45]:
# ufs_ds: xr.Dataset
# ufs_var: str
# ufs_stats: dict
# verif_ds: xr.Dataset
# erif_var: str
# verif_stats: dict
                    
rmses = [stats.calc_rmse_spread(ufs_nao[i], ufs_var_list[i], ufs_stats[i], era5_nao, era5_var, era5_stats)
         for i in range(len(ufs_nao))]

Number of ensemble members: 11
Number of years: 28
Accumulating Statistics...
Finished.
Number of ensemble members: 11
Number of years: 28
Accumulating Statistics...


KeyError: "not all values found in index 'time'. Try setting the `method` keyword argument (example: method='nearest')."

In [ ]:
seas5_rmse = stats.calc_rmse_spread(seas5_nao, seas5_var, seas5_stats, era5_nao, era5_var, era5_stats)

In [ ]:
-----

In [ ]:
ufs_experiments.insert(2, 'seas5')
rmses.insert(2, seas5_rmse)